In [1]:
import pandas as pd
import requests
import urllib3
import os
from pathlib import Path
from urllib3.exceptions import InsecureRequestWarning
import warnings

urllib3.disable_warnings(InsecureRequestWarning)
warnings.simplefilter(action='ignore', category=pd.errors.ParserWarning)

reeds_path = os.path.expanduser('~/Documents/Github/ReEDS/ReEDS/')

In [2]:
state_abbrev_map = {
    'Alabama': 'AL',
    'Alaska': 'AK',
    'Arizona': 'AZ',
    'Arkansas': 'AR',
    'California': 'CA',
    'Colorado': 'CO',
    'Connecticut': 'CT',
    'Delaware': 'DE',
    'District of Columbia': 'DC',
    'Florida': 'FL',
    'Georgia': 'GA',
    'Hawaii': 'HI',
    'Idaho': 'ID',
    'Illinois': 'IL',
    'Indiana': 'IN',
    'Iowa': 'IA',
    'Kansas': 'KS',
    'Kentucky': 'KY',
    'Louisiana': 'LA',
    'Maine': 'ME',
    'Maryland': 'MD',
    'Massachusetts': 'MA',
    'Michigan': 'MI',
    'Minnesota': 'MN',
    'Mississippi': 'MS',
    'Missouri': 'MO',
    'Montana': 'MT',
    'Nebraska': 'NE',
    'Nevada': 'NV',
    'New Hampshire': 'NH',
    'New Jersey': 'NJ',
    'New Mexico': 'NM',
    'New York': 'NY',
    'North Carolina': 'NC', 
    'North Dakota': 'ND',
    'Ohio': 'OH',
    'Oklahoma': 'OK',
    'Oregon': 'OR',
    'Pennsylvania': 'PA',
    'Rhode Island': 'RI',
    'South Carolina': 'SC',
    'South Dakota': 'SD',
    'Tennessee': 'TN',
    'Texas': 'TX',
    'Utah': 'UT',
    'Vermont': 'VT',
    'Virginia': 'VA',
    'Washington': 'WA',
    'West Virginia': 'WV',
    'Wisconsin': 'WI',
    'Wyoming': 'WY'
}
abbrev_state_map = {v:k for k,v in state_abbrev_map.items()}

In [3]:
# Download state- and cendiv-level daily HDD/CDDs 
def download_daily_state_degree_day_data(outdir):
    for dd_type in ['cdd', 'hdd']:
        for year in range(2014, 2025):
            if dd_type == 'cdd':
                energy_type = 'Cooling'
            else:
                energy_type = 'Heating'
    
            url = f"https://ftp.cpc.ncep.noaa.gov/htdocs/degree_days/weighted/daily_data/{year}/StatesCONUS.{energy_type}.txt"
            fpath = Path(outdir, f"daily_state_{dd_type}_{year}.txt")
        
            response = requests.get(url, verify=False)
            if response.status_code == 200:
                with open(fpath, "w", encoding="utf-8") as f:
                    f.write(response.text)
            else:
                print(f"Error: {response.status_code}")

def download_daily_cendiv_degree_day_data(outdir):
    for dd_type in ['cdd', 'hdd']:
        for year in range(2014, 2025):
            if dd_type == 'cdd':
                energy_type = 'Cooling'
            else:
                energy_type = 'Heating'
    
            url = f"https://ftp.cpc.ncep.noaa.gov/htdocs/degree_days/weighted/daily_data/{year}/Population.{energy_type}.txt"
            fpath = Path(outdir, f"daily_cendiv_{dd_type}_{year}.txt")
        
            response = requests.get(url, verify=False)
            if response.status_code == 200:
                with open(fpath, "w", encoding="utf-8") as f:
                    f.write(response.text)
            else:
                print(f"Error: {response.status_code}")


download_daily_state_degree_day_data('inputs')
download_daily_cendiv_degree_day_data('inputs')

In [4]:
def read_historical_state_populations(fpath, year_range):
    hist_population = pd.read_excel(fpath, skiprows=3)
    hist_population = (
        hist_population.rename(columns={'Unnamed: 0': 'state'})
        .set_index('state')
        [year_range]
        .dropna(subset=year_range[0])
        .iloc[5:]
    )
    hist_population.index = hist_population.index.str.replace('.', '')

    return hist_population


hist_populations_2010s = read_historical_state_populations(
    Path('inputs', 'nst-est2020.xlsx'),
    range(2010, 2020)
)
hist_populations_2020s = read_historical_state_populations(
    Path('inputs', 'NST-EST2025-POP.xlsx'),
    range(2020, 2026)
)
hist_populations = pd.concat(
    [hist_populations_2010s, hist_populations_2020s],
    axis=1
)
hist_populations.index = hist_populations.index.map(state_abbrev_map)

In [5]:
gasreg_state_map = {
    'Northwest': ['OR', 'WA'],
    'California': ['CA'],
    'Mountain': ['MT', 'ID', 'WY', 'CO', 'UT', 'NV'],
    'Southwest': ['AZ', 'NM']
}

In [6]:
# Calculate daily HDD/CDDs for gasregs that are not census divisions
def calculate_daily_gasreg_degree_days(dd_type, hist_populations, gasreg_state_map):
    gasreg_dd = []
    for year in range(2014, 2025):
        df = pd.read_csv(
            Path('inputs', f'daily_state_{dd_type}_{year}.txt'),
            sep='|',
            skiprows=3
        )
        df = df.rename(columns={'Region': 'state'}).set_index('state').transpose()
    
        gasreg_dds = {}
        for gasreg, states in gasreg_state_map.items():
            state_weighted_dds = []
            gasreg_population = hist_populations.loc[states].sum()
        
            for state in states:
                weight = hist_populations.loc[state][year] / gasreg_population.loc[year]
                weighted_dd = df[state] * weight
                state_weighted_dds.append(weighted_dd)
        
            gasreg_dds[gasreg] = pd.concat(state_weighted_dds, axis=1).sum(axis=1)
    
        gasreg_dds = pd.concat(gasreg_dds, axis=1)
        gasreg_dd.append(gasreg_dds)
    
    gasreg_dd = pd.concat(gasreg_dd)
    gasreg_dd.index = pd.to_datetime(gasreg_dd.index)
    gasreg_dd = gasreg_dd.set_index([
        gasreg_dd.index.year,
        gasreg_dd.index.month,
        gasreg_dd.index.day
    ])
    gasreg_dd.index = gasreg_dd.index.rename(['year', 'month', 'day'])

    return gasreg_dd

gasreg_cdd = calculate_daily_gasreg_degree_days('cdd', hist_populations, gasreg_state_map)
gasreg_hdd = calculate_daily_gasreg_degree_days('hdd', hist_populations, gasreg_state_map)

In [7]:
# Get daily HDD/CDDs for census divisions
cendiv_number_name_map = {
    '1': 'New_England',
    '2': 'Mid_Atlantic',
    '3': 'East_North_Central',
    '4': 'West_North_Central',
    '5': 'South_Atlantic',
    '6': 'East_South_Central',
    '7': 'West_South_Central',
    '8': 'Mountain',
    '9': 'Pacific',
    'CONUS': 'CONUS'
}

def get_census_division_degree_days(year, dd_type):
    dd = pd.read_csv(
        Path('inputs', f"daily_cendiv_{dd_type}_{year}.txt"),
        sep='|',
        skiprows=3
    )
    dd['Region'] = dd['Region'].map(cendiv_number_name_map)
    dd = (
        dd.dropna(subset='Region')
        .set_index('Region')
        .transpose()
    )
    dd.index = pd.to_datetime(dd.index)

    return dd

hdd_list = []
cdd_list = []
for year in range(2014, 2025):
    hdd_list.append(get_census_division_degree_days(year, 'hdd'))
    cdd_list.append(get_census_division_degree_days(year, 'cdd'))

cendiv_hdd = pd.concat(hdd_list)
cendiv_hdd = cendiv_hdd.set_index([
    cendiv_hdd.index.year,
    cendiv_hdd.index.month,
    cendiv_hdd.index.day
])
cendiv_hdd.index = cendiv_hdd.index.rename(['year', 'month', 'day'])

cendiv_cdd = pd.concat(cdd_list)
cendiv_cdd = cendiv_cdd.set_index([
    cendiv_cdd.index.year,
    cendiv_cdd.index.month,
    cendiv_cdd.index.day
])
cendiv_cdd.index = cendiv_cdd.index.rename(['year', 'month', 'day'])

In [8]:
# Combine to get daily HDD/CDDs for all gasregs
gasreg_hdd = pd.concat([
    cendiv_hdd.drop(columns=['Mountain', 'Pacific', 'CONUS']),
    gasreg_hdd
], axis=1)

gasreg_cdd = pd.concat([
    cendiv_cdd.drop(columns=['Mountain', 'Pacific', 'CONUS']),
    gasreg_cdd
], axis=1)

In [9]:
# Get natural gas hub prices
hub_prices = pd.read_excel(
    '//nrelnas01/ReEDS/FY26_NatGas_KO/Natural Gas Daily Hub Prices - 07-17-2025 - Internal NREL only.xlsx'
)
hub_prices = hub_prices.loc[(
    hub_prices['Delivery Date'].dt.year.isin(range(2014, 2025))
)].copy()

In [11]:
# Assign hubs to gasregs. These are largely based on finding the geographic
# overlap between hub locations and gasregs in inspect_hub_locations.ipynb.
# In some cases, (e.g., East North Central), only a subset of the identified
# hub locations is chosen because it results in a better correlation (i.e.,
# higher HDD/CDD coefficients in the regression)
region_hub_map = {
    'California': ['California - North', 'California - South'],
    'East_North_Central': ['Chicago Metro'],
    'East_South_Central': ['Marcellus - Lower'],
    'Mid_Atlantic': ['Mid-Atlantic'],
    'Mountain': ['Niobrara DJ', 'Green River', 'Piceance/Uinta'],
    'New_England': ['New England'],
    'Northwest': ['Northwest'],
    'South_Atlantic': ['South Atlantic', 'Marcellus - Lower', 'Mid-Atlantic'],
    'Southwest': ['Mojave', 'San Juan'],
    'West_North_Central': ['Midcontinent - Upper', 'Midcontinent - Central'],
    'West_South_Central': [
        'Barnett',
        'Gulf Coast ELA',
        'Gulf Coast ETX',
        'Gulf Coast STX',
        'Gulf Coast WLA',
        'Haynesville',
        'Henry Hub',
        'Midcontinent - Lower',
        'TexOK',
    ]
}

# For each gasreg, calculate the volume-weighted average price of all hubs
def get_regional_volume_weighted_price(hub_prices, region, hub_names):
    hub_names = [f"Hitachi Energy {hub_name}" for hub_name in hub_names]
    select_hub_prices = (
        hub_prices
        .loc[hub_prices['Price Hub'].isin(hub_names)]
        .copy()
        .set_index('Delivery Date')
    )

    if len(hub_names) > 1:
        select_hub_prices['Total $'] = (
            select_hub_prices['Wtd Avg Index $'] * select_hub_prices['Daily Volume']
        )
        select_hub_prices = (
            select_hub_prices.groupby(level=0)
            .sum(numeric_only=True)
        )
        select_hub_prices['volume_weighted_price'] = (
            select_hub_prices['Total $'] / select_hub_prices['Daily Volume']
        )
        regional_volume_weighted_price = select_hub_prices['volume_weighted_price']
    else:
        regional_volume_weighted_price = select_hub_prices['Wtd Avg Index $']

    return regional_volume_weighted_price

regional_prices = {}
for region, hub_names in region_hub_map.items():
    regional_prices[region] = get_regional_volume_weighted_price(
        hub_prices,
        region,
        hub_names
    )

regional_prices = pd.concat(regional_prices, axis=1)
regional_prices = regional_prices.set_index([
    regional_prices.index.year,
    regional_prices.index.month,
    regional_prices.index.day
])
regional_prices = regional_prices.rename_axis(index=['year', 'month', 'day'])

In [12]:
# Combine daily HDD/CDD and gas price data
cdd_data = gasreg_cdd.loc[regional_prices.index].copy()
hdd_data = gasreg_hdd.loc[regional_prices.index].copy()

cdd_data.columns = [f"{col}_cdd" for col in cdd_data.columns]
hdd_data.columns = [f"{col}_hdd" for col in hdd_data.columns]

data = pd.concat([cdd_data, hdd_data], axis=1).fillna(0)
for region in region_hub_map.keys():
    data[f"{region}_price"] = regional_prices[region]

In [13]:
# Export
data.to_csv(Path('inputs', 'gasreg_regression_data.csv'))